In [1]:
import xarray as xr
import pandas as pd
import ast
import panel as pn
from cartopy import crs as ccrs
import matplotlib.pyplot as plt
from dask.distributed import Client
from bokeh.resources import CDN
import xml.etree.ElementTree as ET
import requests
import numpy as np
from os.path import commonpath
from bokeh.models.widgets.tables import HTMLTemplateFormatter
from IPython.display import clear_output
from pathlib import Path
import logging
import dask
import html
import warnings
from tqdm.auto import tqdm
from IPython.core.display import HTML
from bokeh.resources import INLINE
from datasummary_utils import *


pn.extension()
pn.extension('tabulator')

# 1. Suppress Python runtime/deprecation warnings
warnings.filterwarnings('ignore')

# 2. Set Dask logging level to ERROR (silences INFO and WARNING)
dask.config.set({"logging.distributed": "error"})

# 3. Specifically silence the "distributed" logger via Python logging
logging.getLogger("distributed").setLevel(logging.ERROR)

stylesheet = """
/* 1. Base style for all cells */
.tabulator-cell { 
    font-size: 12px; 
}

.tabulator-col-title {
    font-size: 12px !important;
}

"""

def create_data_summ_card(ii, title):
    
    print(ii)
    print(title)
    
    # ids = df.loc[df['institution']==ii]['title'].unique()
    
    dfin = df.loc[df['title']==title]
    
    if len(dfin)>0 :
        summ = create_summary_tabs(dfin, lang=lang, overwrite=overwrite)
        id_str = f'<p style="font-size: 14px;font-weight: bold;">{ii} : {title}</p>'
        card = pn.Card(
            summ,
            collapsed=False,
            collapsible=False,
            header_background='#777877',
            header_color='white',
            header=pn.pane.HTML(id_str, height=20),
            width = max_table_width + 430,
        )
        return card
    return none

def filter_datasets(dfin):
    remove_tit = ['ESPO-R5 v1.0.0 : Ouranos Multipurpose Regional Climate Scenarios']
    dfin = dfin.loc[~dfin['title'].isin(remove_tit)]
    
    return dfin

In [3]:
pn.extension(design='bootstrap')
overwrite = False

indict = {
    "Datasets_1-Climate_Simulations":"dataset_summary_data/simulations.csv",
    "Datasets_3-Reanalysis":"dataset_summary_data/reanalyses.csv",
    "Datasets_2-Observations":"dataset_summary_data/station_obs.csv",
    "Datasets_4-forecasts": "dataset_summary_data/forecasts.csv"
}


# 1. Define your priority list
priority = {"Datasets_1-Climate_Simulations": [r'CRCM5-CMIP6', r'ESPO-G6-R2', r'ESPO-G6-E5L', r'PINS',r'ClimEx', r'CanDCS-M6', r'CanDCS-U6']}

overwrite_html = False # set to true to regnerate all html dataset cards


for html1, csv in indict.items():
    df = correct_institutes(correct_titles(pd.read_csv(csv)))
    df = filter_datasets(df)
    if html1 in priority.keys():
        conditions = []
        choices = []
        for jj, pp in enumerate(priority[html1]):
            conditions.append(df['title'].str.contains(pp, na=False))
            choices.append(jj)
        df['sort_key'] = np.select(conditions, choices, default=3)

        # 4. Sort and drop the temporary key
        df = df.sort_values('sort_key').drop(columns='sort_key')
    
    instit = df['institution'].unique()
    width = max([len(i) for i in instit])
    opts = {}
    for i in instit:
        opts[i] = [l for l in df.loc[df['institution']==i]['title'].unique()]
    for lang in ['en', 'fr']:
        for ii in opts.keys():
            
            for tt in opts[ii]:
                if lang == 'fr':
                    outhtml = f"{tt}_{lang}.html"
                else:
                    outhtml = f"{tt}.html"
                outcard = Path(html1).joinpath(ii, outhtml )
                print(outcard)
                if not outcard.exists() or overwrite_html: 
                    card = create_data_summ_card( ii,tt)
    
                    outcard.parent.mkdir(parents=True, exist_ok=True)
                    card.save(outcard,  embed=True, resources=INLINE)
        clear_output()
print('done')

done
